# `Hutch#`: Stochastic Trace Estimation using Lanczos (Arnoldi?) Decomposition

In [2]:
import jax
import jax.numpy as jnp

from matfree import stochtrace, decomp

In [3]:
_key = lambda x: jax.random.PRNGKey(x)

D = 1_000
rank = 128
num_matvecs = 132
num_samples = 16

def rank1(key, d):
    eps = jax.random.normal(key, (d,1))
    return eps@eps.T

def sum_of_rank1s(key, rank, d):
    keys = jax.random.split(key, rank)
    return jax.vmap(lambda __key: rank1(__key, d))(keys).sum(axis=0)


# Test with a DxD that is probably full rank
# A = jax.random.normal(_key(442), (D,D))
# X = A @ A.T

# Test with a DxD that is a sum of Rank-1 matrices and the identity matrix
X = jnp.eye(D) + sum_of_rank1s(_key(4423), rank, D)

tridiag = decomp.tridiag_sym(num_matvecs)

v = jax.random.normal(_key(22321), (D))

def matvec(v):
    return X@v

res = tridiag(matvec, v)
Q = res.Q_tall
T = res.J_small

head = jnp.trace(T)

# Residual Hutchinson
def proj(v): return v - Q @ (Q.T @ v)
def res_matvec(v): return proj(X @ proj(v))

problem = stochtrace.integrand_trace()
sampler = stochtrace.sampler_normal(v, num=num_samples)
estimate = stochtrace.estimator(problem, sampler)
tail = estimate(res_matvec, key=_key(1243311455143421553))

est = head + tail
true_trace = jnp.trace(X)
print(f"Hutch# trace estimate: {est:.3e}")
print(f"True trace:            {true_trace:.3e}")
print(f"Absolute Diff:         {jnp.abs(est-true_trace):.3e}")

Hutch# trace estimate: 1.287e+05
True trace:            1.292e+05
Absolute Diff:         4.457e+02


In [4]:
problem = stochtrace.integrand_trace()
sampler = stochtrace.sampler_normal(v, num=num_matvecs+num_samples)
estimate = stochtrace.estimator(problem, sampler)
crude_est = estimate(matvec, key=_key(124332163))

print(f"Crude trace estimate:  {crude_est:.3e}")
print(f"True trace:            {true_trace:.3e}")
print(f"Absolute Diff:         {jnp.abs(crude_est-true_trace):.3e}")

Crude trace estimate:  1.275e+05
True trace:            1.292e+05
Absolute Diff:         1.642e+03


## Proof-of-concept comparisons


**Goal:**

- ~~Compare with (crude) Hutchinson~~
- ~~Compare with Hutch++ w/ QR~~

In [8]:
import jax, jax.numpy as jnp
from tqdm.auto import tqdm

# config
D, rank, k, m = 1_000, 10, 15, 15
N_SEEDS, MATRIX_SEED, MODE = 64, 44523, "lowrank"  # "lowrank" or "full"
_key = lambda x: jax.random.PRNGKey(x)

# matrix builders
def _rank1(key, d):
    e = jax.random.normal(key, (d, 1))
    return e @ e.T

def _sum_rank1s(key, r, d):
    keys = jax.random.split(key, r)
    return jax.vmap(lambda kk: _rank1(kk, d))(keys).sum(axis=0)

def make_X(d, r, seed=MATRIX_SEED, mode=MODE):
    if mode == "full":
        A = jax.random.normal(_key(seed), (d, d))
        return A @ A.T
    return jnp.eye(d) + _sum_rank1s(_key(seed), r, d)

# utilities
def apply_cols(Xfun, M):  # (n,s) -> (n,s)
    return jax.vmap(Xfun, in_axes=1, out_axes=1)(M)

# estimators
def lanczos_head(X, k, vkey):
    v0 = jax.random.normal(vkey, (X.shape[0],))
    mv = lambda v: X @ v
    res = decomp.tridiag_sym(k)(mv, v0)
    return jnp.trace(res.J_small), res.Q_tall, v0

def residual_tail(X, Q, m, v0, key):
    mv = lambda v: X @ v
    proj = lambda v: v - Q @ (Q.T @ v)
    res_mv = lambda v: proj(mv(proj(v)))
    problem = stochtrace.integrand_trace()
    est = stochtrace.estimator(problem, stochtrace.sampler_rademacher(v0, num=m))
    return est(res_mv, key=key)

def crude_hutch(X, n, v0, key):
    problem = stochtrace.integrand_trace()
    est = stochtrace.estimator(problem, stochtrace.sampler_rademacher(v0, num=n))
    return est(lambda v: X @ v, key=key)

def hutchpp(Xfun, n, s, key):
    k1, k2 = jax.random.split(key, 2)
    E = jax.random.normal(k1, (n, 2*s))
    S, G = E[:, :s], E[:, s:]
    Y = apply_cols(Xfun, S)                    # (n,s)
    Q, _ = jnp.linalg.qr(Y, mode='reduced')    # (n,s)
    XQ = apply_cols(Xfun, Q)                   # (n,s)
    low = jnp.trace(Q.T @ XQ)
    Gp = G - Q @ (Q.T @ G)
    XGp = apply_cols(Xfun, Gp)
    tail = jnp.trace(Gp.T @ XGp) / s
    return low + tail

# trials
def trial(X, k, m, seed):
    k_v, k_tail, k_crude, k_hpp = jax.random.split(_key(seed), 4)
    head, Q, v0 = lanczos_head(X, k, k_v)
    res_est = head + residual_tail(X, Q, m, v0, k_tail)
    crude = crude_hutch(X, k + m, v0, k_crude)
    s = max(1, (k + m) // 3)
    hpp = hutchpp(lambda v: X @ v, X.shape[0], s, k_hpp)
    return res_est, crude, hpp

def run_many(X, k, m, seeds):
    R, C, H = [], [], []
    for s in tqdm(seeds, desc="seeds"):
        r, c, h = trial(X, k, m, int(s))
        R.append(r); C.append(c); H.append(h)
    return jnp.stack(R), jnp.stack(C), jnp.stack(H)

def summarize(estimates, true_trace):
    err = estimates - true_trace
    return {"mean": estimates.mean(), "std": estimates.std(), "rmse": jnp.sqrt(jnp.mean(err**2))}

# main
if __name__ == "__main__":
    X = make_X(D, rank)
    true_tr = jnp.trace(X)
    seeds = jnp.arange(1, N_SEEDS + 1)

    est_res, est_crude, est_hpp = run_many(X, k, m, seeds)
    s_res, s_crude, s_hpp = summarize(est_res, true_tr), summarize(est_crude, true_tr), summarize(est_hpp, true_tr)

    f = lambda x: {k: float(v) for k, v in x.items()}
    print(f"True trace: {float(true_tr):.6e}")
    print("Residual-Hutch:", f(s_res))
    print("Crude-Hutch   :", f(s_crude))
    print("Hutch++       :", f(s_hpp))
    print(f"Budgets — Residual/Crude/Hutch++ matvecs per trial: {k+m} / {k+m} / {3*max(1,(k+m)//3)}")


seeds: 100%|██████████| 64/64 [00:06<00:00,  9.78it/s]


True trace: 1.111564e+04
Residual-Hutch: {'mean': 11115.859375, 'std': 1.4435603618621826, 'rmse': 1.4599581956863403}
Crude-Hutch   : {'mean': 11154.3046875, 'std': 814.4417724609375, 'rmse': 815.35888671875}
Hutch++       : {'mean': 11114.3056640625, 'std': 91.72362518310547, 'rmse': 91.73332977294922}
Budgets — Residual/Crude/Hutch++ matvecs per trial: 30 / 30 / 30


: 

# Tests with a real GGN

In [19]:
# load MAP state

import optax

from src.utils import load_checkpoint, load_yaml, count_model_params
from src.scalemodels import TrainState, EMPTY_STATS
from src.toymodels import SimpleClassifier
from src.toydata import get_dataloaders

model_name = 'toyclassifier_banana'
cfg_path = f'config/toy/{model_name}.yml'
dataset = 'banana'

cfg = load_yaml(cfg_path)
model_cfg = cfg['model']
opt_cfg = cfg['optimization']
alpha = opt_cfg["alpha"]
map_cfg = opt_cfg["map"]

model_type = model_cfg.get("name", "regressor")  # 'regressor' or 'classifier'
num_h = model_cfg["num_h"]
num_l = model_cfg["num_l"]
num_c = model_cfg.get("num_c", 2) if model_type == "classifier" else 1
rng_model = jax.random.PRNGKey(model_cfg["seed"])
map_batch_size = map_cfg["batch_size"]
epochs_map = map_cfg["epochs"]
lr_map = map_cfg["lr"]

model = SimpleClassifier(numh=num_h, numl=num_l, numc=num_c)

train_loader, test_loader, _ = get_dataloaders(dataset=dataset, batch_size=map_batch_size)

dummy_input = next(iter(train_loader))[0][:1]
variables = model.init(rng_model, dummy_input)
optimizer_map = optax.adam(1e-3)
model_state = TrainState.create(
    apply_fn=model.apply,
    params=variables['params'],
    tx=optimizer_map,
    batch_stats = variables.get('batch_stats', EMPTY_STATS),
)
map_ckpt_prefix = f"map_{dataset}"

map_state = load_checkpoint(
    ckpt_dir="checkpoint/map/",
    prefix=map_ckpt_prefix,
    target=model_state
)

D = count_model_params(variables)

[checkpoint] Loaded model checkpoint from /Users/nielsraunkjaer/Desktop/thesis/laplace-inducing-points/checkpoint/map (prefix=map_banana)


In [20]:
# compute W and GGN

from src.ggn import compute_W_vps, compute_ggn_vp, compute_ggn_dense
from src.utils import flatten_nn_params

flat_params, unravel_fn = flatten_nn_params(map_state.params)

FULL_DATA  = next(iter(train_loader))[0] # 32 samples
# GGN_DATA   = FULL_DATA[:-1]
# EXTRA_TERM = FULL_DATA[None,-1]

GGN_full_dense, *_ =  compute_ggn_dense(map_state, FULL_DATA, model_type=model_type, flat_params=flat_params, unravel_fn=unravel_fn, full_set_size=None)
# GGN_dense, *_      =  compute_ggn_dense(map_state, GGN_DATA,  model_type=model_type, flat_params=flat_params, unravel_fn=unravel_fn, full_set_size=None)

GGN_full =  compute_ggn_vp(map_state, FULL_DATA,  model_type=model_type, flat_params=flat_params, unravel_fn=unravel_fn, full_set_size=None)
# GGN      =  compute_ggn_vp(map_state, GGN_DATA,   model_type=model_type, flat_params=flat_params, unravel_fn=unravel_fn, full_set_size=None)
# W, WT    =  compute_W_vps( map_state, GGN_DATA,   model_type=model_type, flat_params=flat_params, unravel_fn=unravel_fn, full_set_size=None)
# Wp, WTp  =  compute_W_vps( map_state, EXTRA_TERM, model_type=model_type, flat_params=flat_params, unravel_fn=unravel_fn, full_set_size=None)

In [28]:
v = jax.random.normal(_key(69_420_1337_12), (D,))

tridiag = decomp.tridiag_sym(8)

res = tridiag(GGN_full, v)
Q = res.Q_tall
T = res.J_small

#*========*
#* Hutch# *
#*========*
head = jnp.trace(T)

def proj(v): 
    return v - Q @ (Q.T @ v)

def res_matvec(v): 
    t = GGN_full(proj(v))
    return proj(t)

problem = stochtrace.integrand_trace()
sampler = stochtrace.sampler_rademacher(v, num=1)
estimate = stochtrace.estimator(problem, sampler)
tail = estimate(res_matvec, key=_key(1523))

est = head + tail
print("Hutch# trace estimate:", est)
print("True trace:", jnp.trace(GGN_full_dense))

Hutch# trace estimate: 653.3769
True trace: 652.91003


In [26]:
problem = stochtrace.integrand_trace()
sampler = stochtrace.sampler_rademacher(v, num=9) # num_matvecs + num_residual_samples
estimate = stochtrace.estimator(problem, sampler)
crude_est = estimate(GGN_full, key=_key(1211))

print("Crude trace estimate:", crude_est)
print("True trace:", jnp.trace(GGN_full_dense))

Crude trace estimate: 408.8859
True trace: 652.91003
